# H SC 790 · Lab 3 · Write a Phenotype, Then Find Out How Wrong It Is

**Week 4.** Due Thursday, February 18, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- All four measures for all three versions
- Site-level finding identified and explained
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/lmlevine/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

An extract from the Puente County Health Services (PCHS) electronic health record: 16,000 adults, 2024 through 2026, in a Synthea-style layout. Write three versions of a type 2 diabetes phenotype, score each against the chart-review reference standard, overall and by clinic, then find and repair what you discover.

In [ ]:
import duckdb
con = duckdb.connect()
for t in ["patients.csv", "encounters.csv.gz", "conditions.csv.gz", "observations.csv.gz", "medications.csv.gz",
          "facilities.csv", "reference_standard.csv"]:
    con.sql(f"CREATE TABLE {t.split('.')[0]} AS SELECT * FROM read_csv_auto('{COURSE_URL}lab03/{t}')")
con.sql("SHOW TABLES").df()

## 1. Explore the clinical tables

Profile conditions, observations, and medications. Which code systems appear? Which codes, and how often? How many patients have any diabetes-related code?

In [ ]:
# YOUR CODE HERE

## 2. Build your value sets as tables

Type 2 diagnosis codes, type 1 exclusion codes, A1c test codes, and diabetes medications, each as a table with the code, its system, and where the list came from. A value set kept as a table can be reviewed, versioned, and repaired without touching the query.

In [ ]:
# YOUR CODE HERE

## 3. Write three versions of the rule

- **A:** one type 2 diagnosis code.
- **B:** type 2 codes on two or more separate dates.
- **C:** the full rule: a type 2 code, plus a second date, a diabetes medication, or an A1c of 6.5% or higher; type 1 excluded.

Each version is a table of `patient_id`. Version C is given, written against value-set tables named `vs_t2dm_dx`, `vs_t1dm_dx`, `vs_a1c`, and `vs_dm_rx`; rename to match yours.

In [ ]:
PHENO_C = """
WITH dx AS (        -- diagnoses, by distinct date
  SELECT c.patient_id, COUNT(DISTINCT c.onset_date) AS n_dates
  FROM conditions c JOIN vs_t2dm_dx v ON c.code = v.code
  GROUP BY c.patient_id),
lab AS (            -- any A1c at or above threshold
  SELECT DISTINCT o.patient_id
  FROM observations o JOIN vs_a1c v ON o.code = v.code
  WHERE TRY_CAST(o.value AS DOUBLE) >= 6.5),
rx AS (             -- any diabetes medication
  SELECT DISTINCT m.patient_id
  FROM medications m JOIN vs_dm_rx v ON CAST(m.code AS VARCHAR) = CAST(v.code AS VARCHAR)),
t1 AS (             -- exclusion
  SELECT DISTINCT c.patient_id
  FROM conditions c JOIN vs_t1dm_dx v ON c.code = v.code)
SELECT dx.patient_id
FROM dx
WHERE (dx.n_dates >= 2
       OR dx.patient_id IN (SELECT patient_id FROM rx)
       OR dx.patient_id IN (SELECT patient_id FROM lab))
  AND dx.patient_id NOT IN (SELECT patient_id FROM t1)
"""
# YOUR CODE HERE
con.sql("CREATE TABLE pheno_c AS " + PHENO_C)
{t: con.sql(f"SELECT COUNT(*) FROM {t}").fetchall()[0][0] for t in ["pheno_a", "pheno_b", "pheno_c"]}

## 4. Score each against the reference standard

Build the two-by-two table. Report sensitivity, specificity, and both predictive values, with counts. The function below scores one version by clinic (the slide's query); sum over clinics for the overall table.

In [ ]:
def score_by_clinic(pheno_table):
    return con.sql(f"""
      SELECT r.clinic_id,
        COUNT(*) FILTER (WHERE r.truth AND p.patient_id
                         IS NOT NULL)      AS tp,
        COUNT(*) FILTER (WHERE NOT r.truth AND p.patient_id
                         IS NOT NULL)      AS fp,
        COUNT(*) FILTER (WHERE r.truth AND p.patient_id
                         IS NULL)          AS fn,
        COUNT(*) FILTER (WHERE NOT r.truth AND p.patient_id
                         IS NULL)          AS tn
      FROM reference_standard r
      LEFT JOIN {pheno_table} p USING (patient_id)
      GROUP BY r.clinic_id ORDER BY r.clinic_id
    """).df().assign(
        sens=lambda d: d.tp / (d.tp + d.fn),
        ppv =lambda d: d.tp / (d.tp + d.fp))

# YOUR CODE HERE

## 5. Stratify by clinic

Repeat the scoring for each clinic. Find the site where one version behaves differently, and work out why. Counts per clinic are small, so put an interval on what you find before you believe it.

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

**Which clinic, and why?**

*Your answer here.*

## 6. Repair and re-score

Fix the cause by changing a value set or mapping, not the query. Show before and after.

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. Which version would you deploy, and for what use?** Give the measures that justify the choice.

*Your answer here.*

**2. What did version B gain and lose over version A?** State it in patients, not only percentages.

*Your answer here.*

**3. Which clinic differs, and what is the cause?** Show the evidence, then your repair and its effect.

*Your answer here.*

**4. Who would have noticed in production?** With no reference standard, what signal would reveal this failure, and to whom?

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab03/` of your course repository.